## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal(shape=(784, 256), mean=0.0, stddev=0.01))
        self.b1 = tf.Variable(tf.zeros(shape=(256,)))
        self.W2 = tf.Variable(tf.random.normal(shape=(256, 10), mean=0.0, stddev=0.01))
        self.b2 = tf.Variable(tf.zeros(shape=(10,)))


    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # 展平输入: (batch_size, 28, 28) -> (batch_size, 784)
        x = tf.reshape(x, shape=(-1, 784))
        # 隐藏层: ReLU激活
        h = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 输出层: 线性变换（logits）
        logits = tf.matmul(h, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.3029094 ; accuracy 0.18093333
epoch 1 : loss 2.2774634 ; accuracy 0.5596167
epoch 2 : loss 2.2496152 ; accuracy 0.63168335
epoch 3 : loss 2.2156136 ; accuracy 0.65496665
epoch 4 : loss 2.1757925 ; accuracy 0.66641665
epoch 5 : loss 2.1307313 ; accuracy 0.6738
epoch 6 : loss 2.0804992 ; accuracy 0.6811
epoch 7 : loss 2.0251064 ; accuracy 0.6894
epoch 8 : loss 1.9648051 ; accuracy 0.6978667
epoch 9 : loss 1.8999846 ; accuracy 0.7088
epoch 10 : loss 1.8310846 ; accuracy 0.7195333
epoch 11 : loss 1.7586383 ; accuracy 0.73075
epoch 12 : loss 1.6833693 ; accuracy 0.74223334
epoch 13 : loss 1.6060169 ; accuracy 0.75335
epoch 14 : loss 1.5274726 ; accuracy 0.7627
epoch 15 : loss 1.4487307 ; accuracy 0.7711167
epoch 16 : loss 1.3707969 ; accuracy 0.7780833
epoch 17 : loss 1.2946252 ; accuracy 0.78515
epoch 18 : loss 1.2210371 ; accuracy 0.7913833
epoch 19 : loss 1.1507124 ; accuracy 0.79683334
epoch 20 : loss 1.0841955 ; accuracy 0.8017667
epoch 21 : loss 1.0218885 ; accuracy 0